# Provision normalization resources

This notebook uses the **common-first + bounded-parallel fallback** strategy. It downloads pinned common dbSNP BigBeds through resumable segments, queries verified local `/content` copies, and sends only common-missing or common-indeterminate rsIDs to the pinned complete remote indexes with isolated Kent UDC caches. The complete 65/68 GiB files are never downloaded.

The provisioning cell reports worker and segment counts, the durable private JSONL log and checkpoint paths, Drive and local free space, verified legacy-batch reuse, and the privacy boundary. Genotypes are never transmitted or logged; UCSC receives only rsID query identifiers. A rejected common-query leaf is categorized without identifiers and localized; validated sibling checkpoints remain usable, and the authoritative full index resolves both common-missing and common-indeterminate identifiers. An interrupted run is resumable: restart the runtime if needed and rerun the same provisioning cell without deleting prior work. Common-query checkpoints use the verified BigBed content and pinned source identity, not the random `/content` materialization path; compatible legacy checkpoints migrate only after narrow validation, and deterministic common-indeterminate leaves persist for direct authoritative fallback. Completed checksum-valid common caches are reused without consulting mutable HTTP headers. For the current recovery run, set `GENOME_EVIDENCE_DBSNP_WORKERS=2` before rerunning.


In [ ]:
import os
import sys

in_colab = "google.colab" in sys.modules
PROFILE = os.environ.get(
    "GENOME_EVIDENCE_PROFILE", "personal_drive" if in_colab else "synthetic_ci"
)
REPOSITORY_URL = "https://github.com/jcollins-bioinfo/genome-evidence.git"
REPOSITORY_REF = os.environ.get("GENOME_EVIDENCE_GIT_REF", "main")
WORKSPACE_ROOT = os.environ.get(
    "GENOME_EVIDENCE_WORKSPACE", "/content/drive/MyDrive/genome-evidence-private"
)
SUBJECT_ID = os.environ.get("GENOME_EVIDENCE_SUBJECT_ID", "subject-0001")

In [ ]:
import importlib
import importlib.metadata
import json
import subprocess
from hashlib import sha256
from pathlib import Path

if PROFILE not in {"personal_drive", "synthetic_ci"}:
    raise ValueError("PROFILE must be personal_drive or synthetic_ci")

CHECKOUT = Path("/content/genome-evidence-src")
if PROFILE == "personal_drive":
    if "google.colab" in sys.modules:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
    if CHECKOUT.exists():
        remote = subprocess.run(
            ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"],
            check=True,
            capture_output=True,
            text=True,
            timeout=30,
        ).stdout.strip()
        if remote != REPOSITORY_URL:
            raise RuntimeError("Unexpected checkout remote; move the checkout aside and rerun")
        dirty = subprocess.run(
            ["git", "-C", str(CHECKOUT), "status", "--porcelain"],
            check=True,
            capture_output=True,
            text=True,
            timeout=30,
        ).stdout
        if dirty:
            raise RuntimeError("Checkout is dirty; preserve or move it aside and rerun")
    else:
        subprocess.run(
            ["git", "clone", "--no-checkout", REPOSITORY_URL, str(CHECKOUT)],
            check=True,
            timeout=180,
        )
    subprocess.run(
        ["git", "-C", str(CHECKOUT), "fetch", "--force", "origin", REPOSITORY_REF],
        check=True,
        timeout=180,
    )
    RESOLVED_COMMIT = subprocess.run(
        ["git", "-C", str(CHECKOUT), "rev-parse", "--verify", "FETCH_HEAD^{commit}"],
        check=True,
        capture_output=True,
        text=True,
        timeout=30,
    ).stdout.strip()
    loaded_package_modules = [
        module
        for name, module in sys.modules.items()
        if name == "genome_evidence" or name.startswith("genome_evidence.")
    ]
    if loaded_package_modules:
        current_commit = subprocess.run(
            ["git", "-C", str(CHECKOUT), "rev-parse", "HEAD^{commit}"],
            check=True,
            capture_output=True,
            text=True,
            timeout=30,
        ).stdout.strip()
        loaded_paths = [
            Path(str(module_file)).resolve()
            for module in loaded_package_modules
            if (module_file := getattr(module, "__file__", None)) is not None
        ]
        if current_commit != RESOLVED_COMMIT or any(
            not path.is_relative_to(CHECKOUT.resolve()) for path in loaded_paths
        ):
            raise RuntimeError(
                "genome_evidence modules from another revision are already loaded; "
                "restart the runtime and rerun from the first cell"
            )
    subprocess.run(
        ["git", "-C", str(CHECKOUT), "checkout", "--detach", RESOLVED_COMMIT],
        check=True,
        timeout=60,
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "-e",
            f"{CHECKOUT}[notebook]",
        ],
        check=True,
        timeout=600,
    )
    source_root = (CHECKOUT / "src").resolve()
    package_init = source_root / "genome_evidence" / "__init__.py"
    if not package_init.is_file():
        raise RuntimeError("Resolved checkout does not contain the genome_evidence package")
    source_path = str(source_root)
    if source_path not in sys.path:
        sys.path.insert(0, source_path)
    importlib.invalidate_caches()
else:
    RESOLVED_COMMIT = "installed-ci-package"

genome_evidence = importlib.import_module("genome_evidence")
PACKAGE_ORIGIN = Path(genome_evidence.__file__).resolve()
if PROFILE == "personal_drive" and not PACKAGE_ORIGIN.is_relative_to(CHECKOUT.resolve()):
    raise RuntimeError("genome_evidence import origin is outside the resolved checkout")
INSTALLED_VERSION = importlib.metadata.version("genome-evidence")
LOCK_SHA256 = (
    sha256((CHECKOUT / "uv.lock").read_bytes()).hexdigest() if PROFILE == "personal_drive" else None
)
SANITIZED_IMPORT_PATH = (
    str(PACKAGE_ORIGIN.relative_to(CHECKOUT))
    if PROFILE == "personal_drive"
    else "installed-ci-package"
)
BOOTSTRAP_STATUS = {
    "profile": PROFILE,
    "requested_ref": REPOSITORY_REF,
    "resolved_commit": RESOLVED_COMMIT,
    "version": INSTALLED_VERSION,
    "import_path": SANITIZED_IMPORT_PATH,
    "lock_sha256": LOCK_SHA256,
    "lock_equivalent": PROFILE != "personal_drive",
}
print(json.dumps(BOOTSTRAP_STATUS, sort_keys=True))

In [ ]:
from genome_evidence.workspace import WorkspaceConfig, initialize_workspace

if PROFILE == "personal_drive":
    workspace = initialize_workspace(Path(WORKSPACE_ROOT), WorkspaceConfig(subject_id=SUBJECT_ID))
else:
    assert PROFILE == "synthetic_ci"
    workspace = None

In [ ]:
from genome_evidence.workspace import (
    ProvisioningIncomplete,
    provision_personal_normalization_resources,
)

if PROFILE == "personal_drive":
    try:
        result = provision_personal_normalization_resources(
            workspace, os.environ, working_root=Path("/content")
        )
    except ProvisioningIncomplete as error:
        print(
            {
                "status": "incomplete_resumable",
                "message": str(error),
                "log": str(error.log_path.relative_to(workspace)),
                "checkpoint": str(error.checkpoint_path.relative_to(workspace)),
                "next_action": "Rerun this cell; verified completed work will be reused.",
            }
        )
    else:
        print(
            {
                "status": "complete",
                "selection": str(result.selection_path.relative_to(workspace)),
                "source_assembly": result.selection.source_assembly,
                "defined_markers": result.marker_count,
                "cross_build_mapped_markers": result.mapped_marker_count,
                "unresolved_markers": result.unresolved_marker_count,
                "marker_definitions": result.selection.marker_definitions,
                "grch38_fasta": result.selection.grch38_fasta,
                "liftover": result.selection.grch37_to_grch38_liftover,
                "provenance": result.selection.provenance_manifest,
                "log": str(result.log_path.relative_to(workspace)),
                "checkpoint": str(result.checkpoint_path.relative_to(workspace)),
            }
        )
else:
    assert PROFILE == "synthetic_ci"
    print("synthetic_ci: resource provisioning loaded without Drive or network access")

## Provenance and boundaries

The FASTA archive is accepted only when it matches UCSC's published checksum. Marker definitions require exact rsID, assembly, chromosome, and position agreement with the imported source. 23andMe's documented plus-strand convention justifies `orientation=none`; ambiguous, non-rsID, non-SNV, absent, and incompatible cross-build records remain explicitly unresolved. This is research infrastructure, not clinical validation.